# Environment / Dependencies Setup
The cell below installs all required Python packages:

In [ ]:
!pip install -q llama-index-core llama-index-llms-openrouter llama-index-embeddings-huggingface "torch>=2.5" llama-index-readers-file llama-index-retrievers-bm25 python-dotenv pymupdf rank_bm25

### Setup — Import Libraries
Imports the indexing pieces, the dense + sparse retrievers, the fusion tool, and the LLM integration.

In [ ]:
import os

from dotenv import load_dotenv
from llama_index.core import VectorStoreIndex, Settings
from llama_index.core.node_parser import SemanticSplitterNodeParser
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.openrouter import OpenRouter
from llama_index.readers.file import PDFReader
from llama_index.retrievers.bm25 import BM25Retriever

### Setup — Silence Noisy Logs
This cell doesn't touch the pipeline at all — it's purely there to keep the notebook's output readable.

In [ ]:
import logging
import warnings

warnings.filterwarnings("ignore")

# downgrade the chattiest loggers so their messages never reach the notebook
for noisy in ("transformers", "sentence_transformers", "huggingface_hub", "httpx"):
    logging.getLogger(noisy).setLevel(logging.ERROR)

`bm25s`, the library behind `BM25Retriever`, needs one extra step.

In [ ]:
bm25_logger = logging.getLogger("bm25s")
bm25_logger.setLevel(logging.ERROR)
# stop these messages from also bubbling up to the root logger (double output)
bm25_logger.propagate = False

### Setup — Load API Keys & Configure LlamaIndex Settings
This cell gets everything authenticated and configured so every later step knows which LLM and embedding model to use, without being told again.

In [ ]:
load_dotenv(".env")

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = input("Enter your OpenRouter API key (get one at https://openrouter.ai): ").strip()

print("Key loaded.")

# Settings.* are LlamaIndex globals -- every later component reads them automatically
# Set up the LLM (using OpenRouter)
Settings.llm = OpenRouter(
    model="nvidia/nemotron-3-super-120b-a12b:free",
    api_key=OPENROUTER_API_KEY,
    temperature=0,
    max_tokens=512
)

# Set up the dense embedding model
Settings.embed_model = HuggingFaceEmbedding(model_name="all-MiniLM-L6-v2")

print("LlamaIndex configured!")

### Step 1 — Load Documents & Create Semantic Nodes
This step takes the raw PDF and turns it into the meaning-based chunks the rest of the pipeline will search over.

In [ ]:
# Load PDF document
reader = PDFReader()

# One LlamaIndex Document per PDF page
documents = reader.load_data(file="data/sample_text_document.pdf")

print(f"Loaded {len(documents)} document(s)")

# Semantic chunking: splits based on meaning, not fixed size
splitter = SemanticSplitterNodeParser(
    buffer_size=1, 
    # a new chunk starts wherever sentence similarity dips below the 95th percentile
    breakpoint_percentile_threshold=95, 
    embed_model=Settings.embed_model
)

# Parse documents into nodes explicitly for hybrid search
nodes = splitter.get_nodes_from_documents(documents)

print(f"Extracted {len(nodes)} semantic nodes from the documents.")

### Step 2 — Create the Hybrid Retrievers (Dense + Sparse)
Builds a dense retriever, a sparse retriever, and a fused retriever that combines both (flow diagram below).

In [ ]:
# 1. THE DENSE COMPONENT
vector_index = VectorStoreIndex(nodes)
vector_retriever = vector_index.as_retriever(similarity_top_k=3)

# 2. THE SPARSE COMPONENT
bm25_retriever = BM25Retriever.from_defaults(
    nodes=nodes, 
    similarity_top_k=3
)

# 3. THE HYBRID ENGINE (Fusing them together)
hybrid_retriever = QueryFusionRetriever(
    [vector_retriever, bm25_retriever],
    similarity_top_k=3,
    num_queries=1, 
    # reciprocal rank fusion blends the two lists without needing comparable scores
    mode="reciprocal_rerank", 
)

print("Hybrid retrievers (Dense + Sparse) successfully fused!")

### Step 3 — Query the Index
This step is where a real question finally gets asked and answered, using everything built so far.

In [ ]:
# Create a query engine from the fused retriever
query_engine = RetrieverQueryEngine.from_args(hybrid_retriever)

# Ask a question
QUERY = "Why do restoration teams reintroduce tidal flow gradually instead of all at once?"
response = query_engine.query(QUERY)

print(f"Query: {QUERY}")
print(f"\nAnswer: {response}")

### Step 4 — Inspect Post-Fusion Sources
This step doesn't generate anything new — it just reveals what actually went into the answer above.

In [ ]:
# Show the source nodes (retrieved chunks)
print("Source nodes used (Post-Fusion):")
# fusion scores are small rrf fractions (rank-based), not cosine similarities
for i, node in enumerate(response.source_nodes):
    print(f"\n--- Source {i + 1} (score: {node.score:.4f}) ---")
    print(node.text[:200] + "...")